# 🛰️ DynaLand & Deep Learning Spatio-Temporal Anomaly Detection
## Literal "Paper Text" Dataset Reproduction (Sustainability 2023, 15(6), 4725)
**Paper Reference:** Section 4.2.1 Case Studies
- **Altamira:** MODIS MOD13Q1 (250m, NDVI)
- **Brumadinho:** Landsat-8 OLI (30m, NDVI)
- **Mariana:** Sentinel-2 MSI (10m, NDWI)

### Execution Pipeline Overview:
1. **Data Ingestion & Preprocessing**
2. **Step 1: Baselines** (Z-Score Leaky & Leak-Free across all 3 datasets)
3. **Step 2: Altamira Repro** (IF & OCSVM, Leaky & Leak-Free)
4. **Step 3: Brumadinho Repro** (IF & OCSVM, Leaky & Leak-Free)
5. **Step 4: Mariana Repro** (IF & OCSVM, Leaky & Leak-Free)
6. **Step 5: Deep Learning Suite** (Batched GPU LSTM Autoencoder across all 3 datasets)
7. **Step 6: Final Comparative 6-Metric Benchmark Table**
8. **Step 7: Persistent Google Drive Backup**


In [ ]:
import os
import sys

# Detect if executing inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Arkadyg27/TimeSeriesProject.git"
    PROJECT_DIR = "/content/TimeSeriesProject"

    os.chdir('/content')
    if not os.path.exists(PROJECT_DIR):
        os.system(f"git clone {REPO_URL}")
    else:
        os.chdir(PROJECT_DIR)
        os.system("git pull")

    os.chdir(PROJECT_DIR)
    print("Google Colab detected. Working directory set to:", os.getcwd())
else:
    print("Running locally. Working directory set to:", os.getcwd())


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlflow", "rasterio", "pymannkendall", "tabulate"])
else:
    print("Local environment detected. Make sure dependencies are installed.")


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

# 1. Native Colab Authentication
if IN_COLAB:
    try:
        from google.colab import auth
        auth.authenticate_user()
    except Exception as e:
        print('Colab auth notice:', e)

import ee

# Team Earth Engine Projects
PROJECT_IDS = ['889258893131', 'timeseriesproject-503021']

initialized = False
for proj in PROJECT_IDS:
    try:
        ee.Initialize(project=proj)
        print(f'Earth Engine initialized successfully using project: "{proj}"')
        initialized = True
        break
    except Exception:
        continue

if not initialized:
    try:
        ee.Initialize()
        print('Earth Engine initialized using account default project!')
    except Exception:
        print('Prompting interactive Earth Engine authentication...')
        ee.Authenticate()
        ee.Initialize()


In [ ]:
import os, sys, glob, shutil
IN_COLAB = 'google.colab' in sys.modules

# Sync full and partial download checkpoints from Google Drive to preserve download progress
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except ValueError as e:
        if 'Mountpoint must not already contain files' in str(e):
            print('Detected dirty mountpoint. Cleaning up...')
            shutil.rmtree('/content/drive', ignore_errors=True)
            drive.mount('/content/drive', force_remount=True)
        else:
            raise
    
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)

    if os.path.exists(DRIVE_PROJECT_PATH):
        for f in glob.glob(f"{DRIVE_PROJECT_PATH}/*.parquet"):
            dest = f"/content/TimeSeriesProject/{os.path.basename(f)}"
            if not os.path.exists(dest) or os.path.getsize(dest) < 100000:
                shutil.copy2(f, dest)
        print('Smart-Synced dataset caches from Google Drive!')


In [ ]:
%env MLFLOW_ALLOW_FILE_STORE=true
import os, sys
import mlflow
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'
    MLRUNS_DIR = os.path.join(DRIVE_PROJECT_PATH, 'mlruns')
    os.makedirs(MLRUNS_DIR, exist_ok=True)

    # Direct MLflow to log persistently to Google Drive
    os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
    if "MLFLOW_TRACKING_URI" in os.environ:
        del os.environ["MLFLOW_TRACKING_URI"]
    mlflow.set_tracking_uri(f"file:///{MLRUNS_DIR}")

    import IPython
    IPython.get_ipython().run_line_magic('env', f'MLFLOW_TRACKING_URI=file:///{MLRUNS_DIR}')
    print(f"MLflow tracking initialized! Runs will be saved to: {MLRUNS_DIR}")
else:
    print("Local environment: MLflow tracking locally.")


In [ ]:
# Preprocess and center all paper-text datasets (Altamira NDVI, Brumadinho Landsat NDVI, Mariana Sentinel NDWI)
!python run_preprocessing.py --suite paper_text


In [ ]:
# Step 1: Run Baseline Z-Score (Leaky vs. Walk-Forward Leak-Free) on All 3 Paper-Text Datasets
!python run_baseline_all.py --suite paper_text


In [ ]:
# Step 2: Altamira MODIS Reproduction (Isolation Forest + One-Class SVM, Leaky & Leak-Free)
!python Altamira_Modis_repro.py


In [ ]:
# Step 3: Brumadinho Landsat-8 Reproduction (Isolation Forest + One-Class SVM, Leaky & Leak-Free)
!python Brumadinho_Landsat_repro.py


In [ ]:
# Step 4: Mariana Sentinel-2 Reproduction (Isolation Forest + One-Class SVM, Leaky & Leak-Free)
!python Mariana_Sentinel_repro.py


In [ ]:
# Step 5: Train Batched GPU Accelerated Deep LSTM Autoencoder on All 3 Paper-Text Datasets
!python train_deep.py --suite paper_text --batch_size 512


In [ ]:
# Step 6: Compute Standardized 6-Metric Benchmark Table (Baseline, IF, OCSVM, LSTM Autoencoder)
!python compute_custom_metrics.py --suite paper_text


In [ ]:
import os
import pandas as pd

print("=" * 80)
print("             PAPER TEXT SUITE METRICS (Sustainability 15-04725)      ")
print("=" * 80)
if os.path.exists("PAPER_TEXT_ALL_METRICS.csv"):
    df_paper = pd.read_csv("PAPER_TEXT_ALL_METRICS.csv")
    display(df_paper)
else:
    print("PAPER_TEXT_ALL_METRICS.csv not found.")

print("=" * 80)
print("             ARCHIVED SCRIPT DEFAULT SUITE METRICS (DynaLand Code)   ")
print("=" * 80)
if os.path.exists("THREE_DATASETS_ALL_METRICS.csv"):
    df_script = pd.read_csv("THREE_DATASETS_ALL_METRICS.csv")
    display(df_script)
elif os.path.exists("results_archive_script_dataset/THREE_DATASETS_ALL_METRICS.csv"):
    df_script = pd.read_csv("results_archive_script_dataset/THREE_DATASETS_ALL_METRICS.csv")
    display(df_script)
else:
    print("Archived script metrics not found.")


In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DRIVE_BACKUP = "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"
    !mkdir -p "{DRIVE_BACKUP}"
    
    # 1. Back up all Parquet data tables and checkpoints
    !cp -u *.parquet "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 2. Back up Preprocessed matrices
    !cp -r -u Preprocess/ "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 3. Back up GeoTIFF spatial maps
    !cp -r -u Tiff/ "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 4. Back up MLflow tracking runs
    !cp -r -u mlruns/ "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 5. Back up CSV and Markdown metric reports
    !cp -u *.csv *.md "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    print(f"All paper-text datasets, checkpoints, GeoTIFFs, and MLflow runs backed up to: {DRIVE_BACKUP}")
else:
    print("Local execution: All files stored locally.")
